# Лабораторная работа №1: KNN на данных Dota 2

**Студент:** Курносов Максим Юрьевич

**Цель:** построить бейзлайн KNN для задач классификации (предсказание победы Radiant по драфту) и регрессии (предсказание KDA игрока по его статистике).

**Данные:** Dota 2 Pro League Matches (Kaggle), файлы `match.csv` и `players.csv`.

Ноутбук покрывает пункт 2 задания — создание бейзлайна и оценка качества. Улучшение бейзлайна (п. 3) и собственная имплементация (п. 4) — в следующих ноутбуках.

## 1. Проверка окружения и данных

Смотрим, какие файлы доступны в рабочей директории, и проверяем структуру `match.csv` (первые 3 строки).

In [38]:
!ls
!echo ""
!head -n3 match.csv

ability_ids.csv       match.csv		  purchase_log.csv
ability_upgrades.csv  match_outcomes.csv  sample_data
chat.csv	      objectives.csv	  teamfights.csv
cluster_regions.csv   patch_dates.csv	  teamfights_players.csv
dota-2-matches.zip    player_ratings.csv  test_labels.csv
hero_names.csv	      players.csv	  test_player.csv
item_ids.csv	      player_time.csv	  yasp_sample.json

match_id,start_time,duration,tower_status_radiant,tower_status_dire,barracks_status_dire,barracks_status_radiant,first_blood_time,game_mode,radiant_win,negative_votes,positive_votes,cluster
0,1446750112,2375,1982,4,3,63,1,22,True,0,1,155
1,1446753078,2582,0,1846,63,0,221,22,False,0,2,154


## 2. Загрузка датасета

Скачиваем датасет через `kagglehub`. Если скачался zip-архив — распаковываем в `/content/kaggle_dataset`. Выводим содержимое папки, чтобы убедиться, что файлы на месте.

> **Примечание:** в Colab файлы после распаковки лежат в `/content/kaggle_dataset`, но в следующих ячейках используется чтение из текущей директории (`match.csv`, `players.csv`). Если у вас файлы в другом месте — поправьте пути в следующей ячейке.

In [ ]:
import kagglehub
import pandas as pd
import os
import zipfile

# 1. Загрузка датасета
print("Идет загрузка датасета (около 100-200 МБ)...")
path = kagglehub.dataset_download("bwandowando/dota-2-pro-league-matches-2023")
print("Путь к файлам:", path)

# 2. Распаковка
data_path = path
if path.endswith('.zip'):
    print("Обнаружен архив, распаковываем...")
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall('/content/kaggle_dataset')
    data_path = '/content/kaggle_dataset'

print("Содержимое папки:", os.listdir(data_path))


## 3. Загрузка таблиц и сборка признаков

**Классификация (драфт):**
- Из `players.csv` берём по 5 героев каждой команды: `player_slot` 0–4 — Radiant, 128–132 — Dire.
- Сортируем по `player_slot`, чтобы порядок героев был стабильным (r1…r5, d1…d5).
- Через словарь `match_id -> hero_id` мержим героев в `match.csv` по `match_id`.
- Признаки: 10 колонок `r1_hero`…`r5_hero`, `d1_hero`…`d5_hero`. Цель: `radiant_win`.

**Регрессия (KDA):**
- Фильтруем `players.csv` по `leaver_status == 0` (игроки, не покинувшие матч).
- Считаем целевую переменную `kda = (kills + assists) / (deaths + 1)`.
- Признаки: `gold_per_min`, `xp_per_min`, `hero_damage`, `tower_damage`, `level`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, mean_squared_error, mean_absolute_error

# --- 1. ЗАГРУЗКА ---
print("Загрузка match.csv...")
df_match = pd.read_csv('match.csv')

print("Загрузка players.csv...")
df_players = pd.read_csv('players.csv')

# --- 2. СБОРКА ДРАФТА (Feature Engineering) ---
# В players.csv: player_slot 0-4 это Radiant, 128-132 это Dire
# Берем первых 5 героев каждой команды по player_slot

# Фильтруем и сортируем, чтобы порядок героев был всегда одинаковым (1-5)
radiant = df_players[df_players['player_slot'] < 5].sort_values('player_slot')
dire = df_players[df_players['player_slot'] >= 128].sort_values('player_slot')

# Создаем словарь для быстрого мержа
r_dict = radiant.set_index('match_id')['hero_id'].to_dict()
d_dict = dire.set_index('match_id')['hero_id'].to_dict()

# Добавляем колонки в df_match
for i in range(1, 6):
    df_match[f'r{i}_hero'] = df_match['match_id'].map(r_dict).fillna(0).astype(int)
    df_match[f'd{i}_hero'] = df_match['match_id'].map(d_dict).fillna(0).astype(int)

# --- 3. ПОДГОТОВКА ДАННЫХ ---
X_class = df_match[[f'r{i}_hero' for i in range(1, 6)] + [f'd{i}_hero' for i in range(1, 6)]]
y_class = df_match['radiant_win']

# Для регрессии используем players.csv
df_players = df_players[df_players['leaver_status'] == 0]
df_players['kda'] = (df_players['kills'] + df_players['assists']) / (df_players['deaths'] + 1)
regression_features = ['gold_per_min', 'xp_per_min', 'hero_damage', 'tower_damage', 'level']
X_reg = df_players[regression_features]
y_reg = df_players['kda']


Загрузка match.csv...
Загрузка players.csv...


## 4. Бейзлайн: обучение и оценка KNN

**Препроцессинг:**
- `train_test_split` 80/20, `random_state=42`; для классификации — `stratify=y_class` (сохраняем баланс классов).
- `StandardScaler` для обоих наборов (обучаем на train, применяем к test). Для KNN масштабирование критично: алгоритм основан на расстояниях.

**Модели:**
- `KNeighborsClassifier(n_neighbors=5)` с параметрами по умолчанию (`weights='uniform'`, `metric='minkowski'`).
- `KNeighborsRegressor(n_neighbors=5)` с теми же параметрами.

**Метрики:**
- Классификация: Accuracy, ROC-AUC, F1-score.
- Регрессия: RMSE, MAE.

Полученные значения — это результаты бейзлайна, с которыми дальше будет сравниваться улучшенная версия (п. 3 задания).

In [36]:
# --- 4. БЕЙЗЛАЙН ---
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.2, random_state=42, stratify=y_class)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_c_scaled = scaler.fit_transform(X_train_c)
X_test_c_scaled = scaler.transform(X_test_c)
X_train_r_scaled = scaler.fit_transform(X_train_r)
X_test_r_scaled = scaler.transform(X_test_r)

knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train_c_scaled, y_train_c)
y_pred_c = knn_clf.predict(X_test_c_scaled)
y_pred_c_proba = knn_clf.predict_proba(X_test_c_scaled)[:, 1]

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_train_r_scaled, y_train_r)
y_pred_r = knn_reg.predict(X_test_r_scaled)

# --- 5. ОЦЕНКА ---
print("\n=== РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ (KNN Baseline) ===")
print(f"Accuracy: {accuracy_score(y_test_c, y_pred_c):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test_c, y_pred_c_proba):.3f}")
print(f"F1-score: {f1_score(y_test_c, y_pred_c):.3f}")

print("\n=== РЕЗУЛЬТАТЫ РЕГРЕССИИ (KNN Baseline) ===")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.3f}")
print(f"MAE: {mean_absolute_error(y_test_r, y_pred_r):.3f}")


=== РЕЗУЛЬТАТЫ КЛАССИФИКАЦИИ (KNN Baseline) ===
Accuracy: 0.507
ROC-AUC: 0.514
F1-score: 0.537

=== РЕЗУЛЬТАТЫ РЕГРЕССИИ (KNN Baseline) ===
RMSE: 2.422
MAE: 1.361


## 5. Вывод по бейзлайну

- **Классификация:** Accuracy ≈ 0.507, ROC-AUC ≈ 0.514 — на уровне случайного угадывания. Это ожидаемо для KNN на «сырых» `hero_id`: алгоритм трактует номер героя как числовую величину и считает расстояние между героями по их ID, что не отражает реальную близость/синергию героев.
- **Регрессия:** RMSE = 2.422, MAE = 1.361. KDA имеет скошенное распределение с выбросами; KNN-регрессия без обработки выбросов и без категориальных признаков даёт умеренную ошибку.

**Итог:** бейзлайн зафиксирован. Ограничение качества обусловлено прежде всего признаками (raw `hero_id` — слабый признак для KNN), а не гиперпараметрами. Следующий этап — улучшение бейзлайна (п. 3 задания): гипотезы о метрике расстояния, one-hot кодировании героев, обработке выбросов и `weights='distance'`.